In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

PROJECT_ROOT = Path.cwd().parent

TOPO_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "topography"
)

PARCELS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "parcels"
    / "Parcelas_Reto_AGC_CONJUNTO.shp"
)

ELEV_PATH = TOPO_DIR / "Elevacion_INEGI_CEM4_120m.tif"
SLOPE_PATH = TOPO_DIR / "Pendiente_INEGI_CEM4_120m_grados.tif"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ELEV:", ELEV_PATH)
print("SLOPE:", SLOPE_PATH)
print("PARCELS:", PARCELS_PATH)

PROJECT_ROOT: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026
ELEV: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\topography\Elevacion_INEGI_CEM4_120m.tif
SLOPE: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\topography\Pendiente_INEGI_CEM4_120m_grados.tif
PARCELS: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\parcels\Parcelas_Reto_AGC_CONJUNTO.shp


In [3]:
print("=== EXISTENCIA DE ARCHIVOS ===")

print("Elevación existe:", ELEV_PATH.exists())
print("Pendiente existe:", SLOPE_PATH.exists())
print("Parcelas existe:", PARCELS_PATH.exists())

=== EXISTENCIA DE ARCHIVOS ===
Elevación existe: True
Pendiente existe: True
Parcelas existe: True


In [4]:
with rasterio.open(ELEV_PATH) as src:

    print("=== ELEVACIÓN ===")
    print("CRS:", src.crs)
    print("Resolución:", src.res)
    print("Dimensiones:", src.width, "x", src.height)
    print("Bandas:", src.count)
    print("NoData:", src.nodata)
    print("dtype:", src.dtypes[0])
    print("Bounds:", src.bounds)
    print("Transform:", src.transform)

=== ELEVACIÓN ===
CRS: EPSG:6372
Resolución: (120.0, 120.0)
Dimensiones: 2900 x 3362
Bandas: 1
NoData: -9999.0
dtype: float32
Bounds: BoundingBox(left=2715840.0, bottom=650400.0, right=3063840.0, top=1053840.0)
Transform: | 120.00, 0.00, 2715840.00|
| 0.00,-120.00, 1053840.00|
| 0.00, 0.00, 1.00|


In [5]:
with rasterio.open(SLOPE_PATH) as src:

    print("=== PENDIENTE ===")
    print("CRS:", src.crs)
    print("Resolución:", src.res)
    print("Dimensiones:", src.width, "x", src.height)
    print("Bandas:", src.count)
    print("NoData:", src.nodata)
    print("dtype:", src.dtypes[0])
    print("Bounds:", src.bounds)
    print("Transform:", src.transform)

=== PENDIENTE ===
CRS: EPSG:6372
Resolución: (120.0, 120.0)
Dimensiones: 2900 x 3362
Bandas: 1
NoData: -9999.0
dtype: float32
Bounds: BoundingBox(left=2715840.0, bottom=650400.0, right=3063840.0, top=1053840.0)
Transform: | 120.00, 0.00, 2715840.00|
| 0.00,-120.00, 1053840.00|
| 0.00, 0.00, 1.00|


In [6]:
with rasterio.open(ELEV_PATH) as src:

    data = src.read(1)

    valid = data[data != src.nodata]

    print("=== AUDITORÍA DE VALORES — ELEVACIÓN ===")

    print("Píxeles totales:", data.size)
    print("Píxeles válidos:", valid.size)
    print("NoData:", np.sum(data == src.nodata))

    print("\nMínimo:", valid.min())
    print("Máximo:", valid.max())
    print("Media:", valid.mean())
    print("Mediana:", np.median(valid))

    print("\nPercentiles:")
    print(
        pd.Series(valid).quantile(
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        )
    )

    print("\nValores negativos:", np.sum(valid < 0))

=== AUDITORÍA DE VALORES — ELEVACIÓN ===
Píxeles totales: 9749800
Píxeles válidos: 4081208
NoData: 5668592

Mínimo: 8.0
Máximo: 5608.0
Media: 1828.2611
Mediana: 1979.0

Percentiles:
0.01     134.00
0.05     360.00
0.25    1314.00
0.50    1979.00
0.75    2379.00
0.95    2775.00
0.99    3203.93
dtype: float64

Valores negativos: 0


In [7]:
with rasterio.open(SLOPE_PATH) as src:

    data = src.read(1)

    valid = data[data != src.nodata]

    print("=== AUDITORÍA DE VALORES — PENDIENTE ===")

    print("Píxeles totales:", data.size)
    print("Píxeles válidos:", valid.size)
    print("NoData:", np.sum(data == src.nodata))

    print("\nMínimo:", valid.min())
    print("Máximo:", valid.max())
    print("Media:", valid.mean())
    print("Mediana:", np.median(valid))

    print("\nPercentiles:")
    print(
        pd.Series(valid).quantile(
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        )
    )

    print("\nValores negativos:", np.sum(valid < 0))

=== AUDITORÍA DE VALORES — PENDIENTE ===
Píxeles totales: 9749800
Píxeles válidos: 4047765
NoData: 5702035

Mínimo: 0.0
Máximo: 70.63786
Media: 10.171154
Mediana: 7.143111

Percentiles:
0.01     0.119366
0.05     0.477454
0.25     2.565351
0.50     7.143111
0.75    15.501950
0.95    29.337757
0.99    38.499443
dtype: float64

Valores negativos: 0


In [8]:
parcels = gpd.read_file(PARCELS_PATH)

print("=== PARCELAS ===")
print("Filas:", len(parcels))
print("CRS:", parcels.crs)
print("Geometrías:", parcels.geometry.type.value_counts().to_dict())
print("IDs únicos:", parcels["ID_POLIGON"].nunique())

print("\nEstados:")
print(parcels["Estado"].value_counts())

print("\nÁrea (ha):")
print(parcels["área_ha"].describe())

=== PARCELAS ===
Filas: 197
CRS: EPSG:4326
Geometrías: {'Polygon': 197}
IDs únicos: 197

Estados:
Estado
Puebla      100
Hidalgo      51
Tlaxcala     46
Name: count, dtype: int64

Área (ha):
count    197.000000
mean       6.604315
std        7.437631
min        0.350000
25%        2.300000
50%        5.460000
75%        8.090000
max       64.300000
Name: área_ha, dtype: float64


In [9]:
from shapely.geometry import box

In [10]:
with rasterio.open(ELEV_PATH) as src:

    parcels_topo = parcels.to_crs(src.crs)

    raster_box = gpd.GeoSeries(
        [box(*src.bounds)],
        crs=src.crs
    )

    intersects = parcels_topo.geometry.intersects(
        raster_box.iloc[0]
    )

print("=== COBERTURA DEL RASTER ===")
print("Parcelas totales:", len(parcels_topo))
print("Parcelas que intersectan:", intersects.sum())
print("Parcelas fuera:", (~intersects).sum())

=== COBERTURA DEL RASTER ===
Parcelas totales: 197
Parcelas que intersectan: 197
Parcelas fuera: 0


In [11]:
from shapely.geometry import box
from shapely.strtree import STRtree

In [12]:
def extraer_topografia_comparacion(
    raster_path,
    parcels_gdf,
    metric_crs="EPSG:6372"
):
    """
    Extrae valores raster a nivel de parcela mediante tres métodos:

    1. centroid
    2. all_touched
    3. area_weighted

    Se utiliza una ventana raster específica para cada parcela,
    evitando construir geometrías para todo el raster.
    """

    # ---------------------------------------------------------
    # Preparar parcelas en CRS métrico
    # ---------------------------------------------------------

    parcels_metric = parcels_gdf.to_crs(metric_crs).copy()

    resultados = []

    with rasterio.open(raster_path) as src:

        raster_crs = src.crs

        # Verificación importante
        if raster_crs != parcels_metric.crs:
            raise ValueError(
                f"CRS incompatible: raster={raster_crs}, "
                f"parcelas={parcels_metric.crs}"
            )

        nodata = src.nodata
        transform = src.transform

        # -----------------------------------------------------
        # Procesar parcela por parcela
        # -----------------------------------------------------

        for _, parcel in parcels_metric.iterrows():

            parcel_id = parcel["ID_POLIGON"]
            geom = parcel.geometry

            # -------------------------------------------------
            # 1. Ventana mínima que contiene la parcela
            # -------------------------------------------------

            minx, miny, maxx, maxy = geom.bounds

            window = from_bounds(
                minx,
                miny,
                maxx,
                maxy,
                transform=transform
            )

            # Ajustar a límites enteros de píxel
            window = window.round_offsets().round_lengths()

            # Limitar ventana a raster
            window = window.intersection(
                rasterio.windows.Window(
                    0,
                    0,
                    src.width,
                    src.height
                )
            )

            if window.width <= 0 or window.height <= 0:
                resultados.append({
                    "ID_POLIGON": parcel_id,
                    "centroid": np.nan,
                    "all_touched": np.nan,
                    "area_weighted": np.nan,
                    "n_pixeles": 0,
                    "suma_pesos": np.nan
                })
                continue

            # -------------------------------------------------
            # 2. Leer solamente la ventana
            # -------------------------------------------------

            data = src.read(1, window=window)

            row_start = int(window.row_off)
            col_start = int(window.col_off)

            height, width = data.shape

            # -------------------------------------------------
            # 3. Construir polígonos de los píxeles candidatos
            # -------------------------------------------------

            pixel_data = []

            for local_row in range(height):

                for local_col in range(width):

                    value = data[local_row, local_col]

                    if nodata is not None and np.isclose(
                        value,
                        nodata
                    ):
                        continue

                    global_row = row_start + local_row
                    global_col = col_start + local_col

                    x1, y1 = rasterio.transform.xy(
                        transform,
                        global_row,
                        global_col,
                        offset="ul"
                    )

                    x2, y2 = rasterio.transform.xy(
                        transform,
                        global_row,
                        global_col,
                        offset="lr"
                    )

                    pixel_geom = box(
                        min(x1, x2),
                        min(y1, y2),
                        max(x1, x2),
                        max(y1, y2)
                    )

                    pixel_data.append(
                        {
                            "row": global_row,
                            "col": global_col,
                            "value": float(value),
                            "geometry": pixel_geom
                        }
                    )

            # -------------------------------------------------
            # No hay píxeles válidos
            # -------------------------------------------------

            if not pixel_data:

                resultados.append({
                    "ID_POLIGON": parcel_id,
                    "centroid": np.nan,
                    "all_touched": np.nan,
                    "area_weighted": np.nan,
                    "n_pixeles": 0,
                    "suma_pesos": np.nan
                })

                continue

            # -------------------------------------------------
            # 4. Centroid
            # -------------------------------------------------

            centroid_point = geom.centroid

            centroid_value = np.nan

            for p in pixel_data:

                if p["geometry"].covers(centroid_point):

                    centroid_value = p["value"]
                    break

            # -------------------------------------------------
            # 5. All touched
            # -------------------------------------------------

            touched_values = []

            for p in pixel_data:

                if p["geometry"].intersects(geom):

                    touched_values.append(p["value"])

            all_touched_value = (
                np.mean(touched_values)
                if touched_values
                else np.nan
            )

            # -------------------------------------------------
            # 6. Area weighted
            # -------------------------------------------------

            weighted_values = []
            intersection_areas = []

            parcel_area = geom.area

            for p in pixel_data:

                intersection = p["geometry"].intersection(geom)

                if not intersection.is_empty:

                    intersection_area = intersection.area

                    if intersection_area > 0:

                        weighted_values.append(
                            p["value"]
                        )

                        intersection_areas.append(
                            intersection_area
                        )

            if intersection_areas:

                intersection_areas = np.array(
                    intersection_areas,
                    dtype=float
                )

                weighted_values = np.array(
                    weighted_values,
                    dtype=float
                )

                weights = (
                    intersection_areas /
                    parcel_area
                )

                area_weighted_value = np.sum(
                    weighted_values * weights
                )

                suma_pesos = weights.sum()

            else:

                area_weighted_value = np.nan
                suma_pesos = np.nan

            # -------------------------------------------------
            # 7. Guardar resultado
            # -------------------------------------------------

            resultados.append({
                "ID_POLIGON": parcel_id,
                "centroid": centroid_value,
                "all_touched": all_touched_value,
                "area_weighted": area_weighted_value,
                "n_pixeles": len(intersection_areas)
                    if intersection_areas is not None
                    and len(intersection_areas) > 0
                    else 0,
                "suma_pesos": suma_pesos
            })

    return pd.DataFrame(resultados)

In [13]:
from rasterio.windows import from_bounds

In [14]:
test_parcel = parcels.iloc[[0]].copy()

print(test_parcel[
    ["ID_POLIGON", "área_ha"]
])

  ID_POLIGON  área_ha
0    AGC_196     8.57


In [15]:
test_elevacion = extraer_topografia_comparacion(
    ELEV_PATH,
    test_parcel
)

print(test_elevacion)

  ID_POLIGON  centroid  all_touched  area_weighted  n_pixeles  suma_pesos
0    AGC_196    2542.0  2545.571429    1871.202567          7    0.735484


In [16]:
print("\n=== VALIDACIÓN TEST ===")

print("Filas:", len(test_elevacion))
print(
    "ID único:",
    test_elevacion["ID_POLIGON"].nunique()
)

print(
    "Nulos:",
    test_elevacion[
        ["centroid", "all_touched", "area_weighted"]
    ].isna().sum()
)

print(
    "Suma pesos:",
    test_elevacion["suma_pesos"].iloc[0]
)

print(
    "Número de píxeles:",
    test_elevacion["n_pixeles"].iloc[0]
)


=== VALIDACIÓN TEST ===
Filas: 1
ID único: 1
Nulos: centroid         0
all_touched      0
area_weighted    0
dtype: int64
Suma pesos: 0.7354844421297414
Número de píxeles: 7


In [17]:
parcel_test_metric = test_parcel.to_crs("EPSG:6372")

geom = parcel_test_metric.geometry.iloc[0]

print("=== AGC_196 ===")
print("Área shapefile (ha):", test_parcel["área_ha"].iloc[0])
print("Área geométrica (m²):", geom.area)
print("Área geométrica (ha):", geom.area / 10000)
print("Bounds:", geom.bounds)

=== AGC_196 ===
Área shapefile (ha): 8.57
Área geométrica (m²): 85130.66878760715
Área geométrica (ha): 8.513066878760716
Bounds: (2857386.322475705, 854855.7850213784, 2857755.712358307, 855204.3150285732)


In [18]:
with rasterio.open(ELEV_PATH) as src:

    minx, miny, maxx, maxy = geom.bounds

    window = from_bounds(
        minx,
        miny,
        maxx,
        maxy,
        transform=src.transform
    )

    window = window.round_offsets().round_lengths()

    print("=== WINDOW ===")
    print("Window:", window)
    print("Width:", window.width)
    print("Height:", window.height)

    data = src.read(1, window=window)

    print("\nData shape:", data.shape)
    print("NoData:", src.nodata)

    valid_mask = data != src.nodata

    print("Píxeles candidatos:", data.size)
    print("Píxeles válidos:", valid_mask.sum())
    print("Píxeles NoData:", (~valid_mask).sum())

=== WINDOW ===
Window: Window(col_off=1179, row_off=1655, width=3, height=3)
Width: 3
Height: 3

Data shape: (3, 3)
NoData: -9999.0
Píxeles candidatos: 9
Píxeles válidos: 9
Píxeles NoData: 0


In [19]:
with rasterio.open(ELEV_PATH) as src:

    transform = src.transform
    nodata = src.nodata

    total_intersection_area = 0.0
    valid_intersection_area = 0.0
    nodata_intersection_area = 0.0

    total_pixels_intersected = 0
    valid_pixels_intersected = 0
    nodata_pixels_intersected = 0

    for local_row in range(data.shape[0]):

        for local_col in range(data.shape[1]):

            global_row = int(window.row_off) + local_row
            global_col = int(window.col_off) + local_col

            x1, y1 = rasterio.transform.xy(
                transform,
                global_row,
                global_col,
                offset="ul"
            )

            x2, y2 = rasterio.transform.xy(
                transform,
                global_row,
                global_col,
                offset="lr"
            )

            pixel_geom = box(
                min(x1, x2),
                min(y1, y2),
                max(x1, x2),
                max(y1, y2)
            )

            intersection = pixel_geom.intersection(geom)

            if intersection.is_empty:
                continue

            intersection_area = intersection.area

            if intersection_area <= 0:
                continue

            total_intersection_area += intersection_area
            total_pixels_intersected += 1

            value = data[local_row, local_col]

            if value == nodata:
                nodata_intersection_area += intersection_area
                nodata_pixels_intersected += 1
            else:
                valid_intersection_area += intersection_area
                valid_pixels_intersected += 1

    print("=== COBERTURA REAL AGC_196 ===")

    print("Área parcela:", geom.area)

    print("\nÁrea total intersectada:", total_intersection_area)

    print(
        "Diferencia contra parcela:",
        total_intersection_area - geom.area
    )

    print("\nÁrea sobre píxeles válidos:", valid_intersection_area)

    print("Área sobre NoData:", nodata_intersection_area)

    print(
        "\nCobertura válida:",
        valid_intersection_area / geom.area
    )

    print(
        "Cobertura NoData:",
        nodata_intersection_area / geom.area
    )

    print("\nPíxeles intersectados:", total_pixels_intersected)
    print("Píxeles válidos:", valid_pixels_intersected)
    print("Píxeles NoData:", nodata_pixels_intersected)

=== COBERTURA REAL AGC_196 ===
Área parcela: 85130.66878760715

Área total intersectada: 62612.28244138504
Diferencia contra parcela: -22518.386346222112

Área sobre píxeles válidos: 62612.28244138504
Área sobre NoData: 0.0

Cobertura válida: 0.7354844421297414
Cobertura NoData: 0.0

Píxeles intersectados: 7
Píxeles válidos: 7
Píxeles NoData: 0


In [20]:
import math
from rasterio.windows import Window

In [21]:
with rasterio.open(ELEV_PATH) as src:

    minx, miny, maxx, maxy = geom.bounds

    raw_window = from_bounds(
        minx,
        miny,
        maxx,
        maxy,
        transform=src.transform
    )

    col_start = math.floor(raw_window.col_off)
    row_start = math.floor(raw_window.row_off)

    col_end = math.ceil(
        raw_window.col_off + raw_window.width
    )

    row_end = math.ceil(
        raw_window.row_off + raw_window.height
    )

    corrected_window = Window(
        col_start,
        row_start,
        col_end - col_start,
        row_end - row_start
    )

    print("=== WINDOW ORIGINAL ===")
    print(raw_window)

    print("\n=== WINDOW CORREGIDA ===")
    print(corrected_window)

    print("\nAncho píxeles:", corrected_window.width)
    print("Alto píxeles:", corrected_window.height)

=== WINDOW ORIGINAL ===
Window(col_off=np.float64(1179.5526872975402), row_off=np.float64(1655.2973747618903), width=np.float64(3.0782490216856786), height=np.float64(2.904416726622685))

=== WINDOW CORREGIDA ===
Window(col_off=1179, row_off=1655, width=4, height=4)

Ancho píxeles: 4
Alto píxeles: 4


In [22]:
with rasterio.open(ELEV_PATH) as src:

    total_intersection_area = 0.0
    pixel_count = 0

    for local_row in range(
        int(corrected_window.height)
    ):

        for local_col in range(
            int(corrected_window.width)
        ):

            global_row = (
                int(corrected_window.row_off)
                + local_row
            )

            global_col = (
                int(corrected_window.col_off)
                + local_col
            )

            x1, y1 = rasterio.transform.xy(
                src.transform,
                global_row,
                global_col,
                offset="ul"
            )

            x2, y2 = rasterio.transform.xy(
                src.transform,
                global_row,
                global_col,
                offset="lr"
            )

            pixel_geom = box(
                min(x1, x2),
                min(y1, y2),
                max(x1, x2),
                max(y1, y2)
            )

            intersection = pixel_geom.intersection(
                geom
            )

            if not intersection.is_empty:

                area = intersection.area

                if area > 0:
                    total_intersection_area += area
                    pixel_count += 1

    print("=== VALIDACIÓN DE COBERTURA ===")

    print(
        "Área parcela:",
        geom.area
    )

    print(
        "Área total intersectada:",
        total_intersection_area
    )

    print(
        "Diferencia:",
        total_intersection_area - geom.area
    )

    print(
        "Error relativo:",
        abs(
            total_intersection_area - geom.area
        ) / geom.area
    )

    print(
        "Píxeles intersectados:",
        pixel_count
    )

=== VALIDACIÓN DE COBERTURA ===
Área parcela: 85130.66878760715
Área total intersectada: 85130.66878762915
Diferencia: 2.200249582529068e-08
Error relativo: 2.584555735158712e-13
Píxeles intersectados: 13


In [23]:
# ============================================================
# EXTRACCIÓN TOPOGRÁFICA A NIVEL DE PARCELA
# Comparación:
#   1. Centroid
#   2. All Touched
#   3. Area Weighted
#
# Diseño:
# - Procesamiento por ventana raster
# - No se construye todo el grid de 9.7 millones de píxeles
# - Cálculo de áreas en CRS proyectado
# - Control explícito de NoData y cobertura
# ============================================================

import math
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from shapely.geometry import box
from rasterio.windows import from_bounds, Window


def extraer_topografia_comparacion(
    raster_path,
    parcels_gdf
):
    """
    Extrae un raster topográfico a nivel de parcela mediante
    tres métodos espaciales:

        1. centroid
        2. all_touched
        3. area_weighted

    La extracción se realiza por ventanas raster específicas
    para cada parcela.

    Parameters
    ----------
    raster_path : str o Path
        Ruta al raster GeoTIFF.

    parcels_gdf : GeoDataFrame
        GeoDataFrame de parcelas con columna ID_POLIGON.

    Returns
    -------
    pandas.DataFrame
        Resultados por parcela.

    Notas
    -----
    - Las áreas se calculan en el CRS del raster.
    - Se requiere que el CRS del raster sea proyectado.
    - No se normalizan artificialmente los pesos si existe
      cobertura parcial por NoData.
    """

    # ========================================================
    # 1. Abrir raster y preparar CRS
    # ========================================================

    resultados = []

    with rasterio.open(raster_path) as src:

        raster_crs = src.crs
        transform = src.transform
        nodata = src.nodata

        if raster_crs is None:
            raise ValueError(
                "El raster no tiene CRS definido."
            )

        # ----------------------------------------------------
        # Para calcular áreas necesitamos unidades métricas.
        # El raster topográfico está en EPSG:6372.
        # ----------------------------------------------------

        if not raster_crs.is_projected:
            raise ValueError(
                f"El raster no está en un CRS proyectado: "
                f"{raster_crs}"
            )

        # ----------------------------------------------------
        # Reproyectar parcelas al CRS del raster.
        # Esto NO modifica el archivo original.
        # ----------------------------------------------------

        parcels_metric = parcels_gdf.to_crs(
            raster_crs
        ).copy()

        # ====================================================
        # 2. Procesar cada parcela
        # ====================================================

        for _, parcel in parcels_metric.iterrows():

            parcel_id = parcel["ID_POLIGON"]
            geom = parcel.geometry

            # ------------------------------------------------
            # Validación geométrica básica
            # ------------------------------------------------

            if geom is None or geom.is_empty:

                resultados.append({
                    "ID_POLIGON": parcel_id,
                    "centroid": np.nan,
                    "all_touched": np.nan,
                    "area_weighted": np.nan,
                    "n_pixeles": 0,
                    "suma_pesos": np.nan,
                    "area_intersecciones_m2": 0.0,
                    "area_parcela_m2": np.nan,
                    "cobertura_valida": np.nan
                })

                continue

            parcel_area = geom.area

            if parcel_area <= 0:

                resultados.append({
                    "ID_POLIGON": parcel_id,
                    "centroid": np.nan,
                    "all_touched": np.nan,
                    "area_weighted": np.nan,
                    "n_pixeles": 0,
                    "suma_pesos": np.nan,
                    "area_intersecciones_m2": 0.0,
                    "area_parcela_m2": parcel_area,
                    "cobertura_valida": np.nan
                })

                continue

            # =================================================
            # 3. Crear ventana raster que cubra COMPLETAMENTE
            #    el bounding box de la parcela
            # =================================================

            minx, miny, maxx, maxy = geom.bounds

            raw_window = from_bounds(
                minx,
                miny,
                maxx,
                maxy,
                transform=transform
            )

            # -------------------------------------------------
            # Expandimos la ventana:
            #
            # inicio -> floor
            # final  -> ceil
            #
            # Esto evita cortar parte de la parcela.
            # -------------------------------------------------

            col_start = math.floor(
                raw_window.col_off
            )

            row_start = math.floor(
                raw_window.row_off
            )

            col_end = math.ceil(
                raw_window.col_off +
                raw_window.width
            )

            row_end = math.ceil(
                raw_window.row_off +
                raw_window.height
            )

            window = Window(
                col_start,
                row_start,
                col_end - col_start,
                row_end - row_start
            )

            # -------------------------------------------------
            # Limitar ventana a los límites del raster
            # -------------------------------------------------

            raster_window = Window(
                0,
                0,
                src.width,
                src.height
            )

            window = window.intersection(
                raster_window
            )

            if (
                window.width <= 0
                or window.height <= 0
            ):

                resultados.append({
                    "ID_POLIGON": parcel_id,
                    "centroid": np.nan,
                    "all_touched": np.nan,
                    "area_weighted": np.nan,
                    "n_pixeles": 0,
                    "suma_pesos": np.nan,
                    "area_intersecciones_m2": 0.0,
                    "area_parcela_m2": parcel_area,
                    "cobertura_valida": 0.0
                })

                continue

            # =================================================
            # 4. Leer solamente la ventana
            # =================================================

            data = src.read(
                1,
                window=window
            )

            # Transform del subset leído.
            window_transform = src.window_transform(
                window
            )

            # =================================================
            # 5. Construir únicamente los píxeles candidatos
            # =================================================

            pixel_data = []

            height, width = data.shape

            for local_row in range(height):

                for local_col in range(width):

                    value = data[
                        local_row,
                        local_col
                    ]

                    # -----------------------------------------
                    # Identificar NoData
                    # -----------------------------------------

                    if nodata is not None:

                        if np.isclose(
                            value,
                            nodata
                        ):
                            continue

                    # -----------------------------------------
                    # Evitar NaN
                    # -----------------------------------------

                    if not np.isfinite(value):
                        continue

                    # -----------------------------------------
                    # Geometría del píxel local
                    # -----------------------------------------

                    x1, y1 = rasterio.transform.xy(
                        window_transform,
                        local_row,
                        local_col,
                        offset="ul"
                    )

                    x2, y2 = rasterio.transform.xy(
                        window_transform,
                        local_row,
                        local_col,
                        offset="lr"
                    )

                    pixel_geom = box(
                        min(x1, x2),
                        min(y1, y2),
                        max(x1, x2),
                        max(y1, y2)
                    )

                    pixel_data.append({
                        "value": float(value),
                        "geometry": pixel_geom
                    })

            # =================================================
            # 6. Centroide
            # =================================================

            centroid_point = geom.centroid

            centroid_value = np.nan

            for pixel in pixel_data:

                if pixel["geometry"].covers(
                    centroid_point
                ):

                    centroid_value = (
                        pixel["value"]
                    )

                    break

            # =================================================
            # 7. All Touched
            # =================================================

            touched_values = []

            for pixel in pixel_data:

                if pixel["geometry"].intersects(
                    geom
                ):

                    touched_values.append(
                        pixel["value"]
                    )

            if touched_values:

                all_touched_value = float(
                    np.mean(touched_values)
                )

            else:

                all_touched_value = np.nan

            # =================================================
            # 8. Area Weighted
            # =================================================

            weighted_values = []
            intersection_areas = []

            for pixel in pixel_data:

                intersection = (
                    pixel["geometry"]
                    .intersection(geom)
                )

                if intersection.is_empty:
                    continue

                intersection_area = (
                    intersection.area
                )

                # Solo intersecciones con área positiva.
                if intersection_area <= 0:
                    continue

                weighted_values.append(
                    pixel["value"]
                )

                intersection_areas.append(
                    intersection_area
                )

            # =================================================
            # 9. Calcular ponderación espacial
            # =================================================

            if intersection_areas:

                intersection_areas = np.asarray(
                    intersection_areas,
                    dtype=float
                )

                weighted_values = np.asarray(
                    weighted_values,
                    dtype=float
                )

                area_intersections = (
                    intersection_areas.sum()
                )

                weights = (
                    intersection_areas /
                    parcel_area
                )

                area_weighted_value = float(
                    np.sum(
                        weighted_values *
                        weights
                    )
                )

                suma_pesos = float(
                    weights.sum()
                )

                cobertura_valida = float(
                    area_intersections /
                    parcel_area
                )

                n_pixeles = len(
                    intersection_areas
                )

            else:

                area_weighted_value = np.nan
                suma_pesos = np.nan
                area_intersections = 0.0
                cobertura_valida = 0.0
                n_pixeles = 0

            # =================================================
            # 10. Guardar resultado
            # =================================================

            resultados.append({
                "ID_POLIGON": parcel_id,
                "centroid": centroid_value,
                "all_touched": all_touched_value,
                "area_weighted": area_weighted_value,
                "n_pixeles": n_pixeles,
                "suma_pesos": suma_pesos,
                "area_intersecciones_m2": (
                    area_intersections
                ),
                "area_parcela_m2": parcel_area,
                "cobertura_valida": (
                    cobertura_valida
                )
            })

    # ========================================================
    # 11. Convertir resultados a DataFrame
    # ========================================================

    resultado_df = pd.DataFrame(
        resultados
    )

    return resultado_df

In [24]:
test_elevacion = extraer_topografia_comparacion(
    ELEV_PATH,
    test_parcel
)

print(test_elevacion)

  ID_POLIGON  centroid  all_touched  area_weighted  n_pixeles  suma_pesos  \
0    AGC_196    2542.0  2544.769231    2543.659752         13         1.0   

   area_intersecciones_m2  area_parcela_m2  cobertura_valida  
0            85130.668788     85130.668788               1.0  


In [25]:
print("=== VALIDACIÓN AGC_196 ===")

print("\nFilas:")
print(len(test_elevacion))

print("\nID únicos:")
print(test_elevacion["ID_POLIGON"].nunique())

print("\nValores:")
print(
    test_elevacion[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ]
)

print("\nMetadatos geométricos:")
print(
    test_elevacion[
        [
            "n_pixeles",
            "suma_pesos",
            "area_intersecciones_m2",
            "area_parcela_m2",
            "cobertura_valida"
        ]
    ]
)

print("\nNulos:")
print(
    test_elevacion[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].isna().sum()
)

print("\nError de cobertura:")
print(
    abs(
        test_elevacion["area_intersecciones_m2"]
        - test_elevacion["area_parcela_m2"]
    ).iloc[0]
)

print("\nError de suma de pesos:")
print(
    abs(
        test_elevacion["suma_pesos"]
        - 1
    ).iloc[0]
)

=== VALIDACIÓN AGC_196 ===

Filas:
1

ID únicos:
1

Valores:
   centroid  all_touched  area_weighted
0    2542.0  2544.769231    2543.659752

Metadatos geométricos:
   n_pixeles  suma_pesos  area_intersecciones_m2  area_parcela_m2  \
0         13         1.0            85130.668788     85130.668788   

   cobertura_valida  
0               1.0  

Nulos:
centroid         0
all_touched      0
area_weighted    0
dtype: int64

Error de cobertura:
2.200249582529068e-08

Error de suma de pesos:
2.5845992013273644e-13


In [26]:
small_idx = parcels["área_ha"].idxmin()
large_idx = parcels["área_ha"].idxmax()

test_small = parcels.loc[[small_idx]].copy()
test_large = parcels.loc[[large_idx]].copy()

print("=== PARCELA PEQUEÑA ===")
print(
    test_small[
        ["ID_POLIGON", "área_ha"]
    ]
)

print("\n=== PARCELA INTERMEDIA ===")
print(
    test_parcel[
        ["ID_POLIGON", "área_ha"]
    ]
)

print("\n=== PARCELA GRANDE ===")
print(
    test_large[
        ["ID_POLIGON", "área_ha"]
    ]
)

=== PARCELA PEQUEÑA ===
   ID_POLIGON  área_ha
52    AGC_091     0.35

=== PARCELA INTERMEDIA ===
  ID_POLIGON  área_ha
0    AGC_196     8.57

=== PARCELA GRANDE ===
   ID_POLIGON  área_ha
12    AGC_047     64.3


In [27]:
test_small_result = extraer_topografia_comparacion(
    ELEV_PATH,
    test_small
)

test_large_result = extraer_topografia_comparacion(
    ELEV_PATH,
    test_large
)

print("=== PEQUEÑA ===")
print(test_small_result)

print("\n=== GRANDE ===")
print(test_large_result)

=== PEQUEÑA ===
  ID_POLIGON  centroid  all_touched  area_weighted  n_pixeles  suma_pesos  \
0    AGC_091    2636.0       2636.0         2636.0          1         1.0   

   area_intersecciones_m2  area_parcela_m2  cobertura_valida  
0              3492.57896       3492.57896               1.0  

=== GRANDE ===
  ID_POLIGON  centroid  all_touched  area_weighted  n_pixeles  suma_pesos  \
0    AGC_047    2530.0  2527.770492    2527.960031         61         1.0   

   area_intersecciones_m2  area_parcela_m2  cobertura_valida  
0            638659.00459     638659.00459               1.0  


In [28]:
tests_topografia = pd.concat(
    [
        test_small_result,
        test_elevacion,
        test_large_result
    ],
    ignore_index=True
)

tests_topografia["error_area_m2"] = (
    tests_topografia["area_intersecciones_m2"]
    - tests_topografia["area_parcela_m2"]
).abs()

tests_topografia["error_pesos"] = (
    tests_topografia["suma_pesos"] - 1
).abs()

print(
    tests_topografia[
        [
            "ID_POLIGON",
            "area_parcela_m2",
            "n_pixeles",
            "centroid",
            "all_touched",
            "area_weighted",
            "cobertura_valida",
            "error_area_m2",
            "error_pesos"
        ]
    ].to_string(index=False)
)

ID_POLIGON  area_parcela_m2  n_pixeles  centroid  all_touched  area_weighted  cobertura_valida  error_area_m2  error_pesos
   AGC_091      3492.578960          1    2636.0  2636.000000    2636.000000               1.0   4.547474e-13 1.110223e-16
   AGC_196     85130.668788         13    2542.0  2544.769231    2543.659752               1.0   2.200250e-08 2.584599e-13
   AGC_047    638659.004590         61    2530.0  2527.770492    2527.960031               1.0   4.004687e-08 6.283862e-14


In [29]:
print("=== PROCESANDO ELEVACIÓN — 197 PARCELAS ===")

elevacion_comparacion = extraer_topografia_comparacion(
    ELEV_PATH,
    parcels
)

print("\nShape:", elevacion_comparacion.shape)
print(
    "Parcelas:",
    elevacion_comparacion["ID_POLIGON"].nunique()
)

print(
    "Nulos:",
    elevacion_comparacion[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].isna().sum()
)

=== PROCESANDO ELEVACIÓN — 197 PARCELAS ===

Shape: (197, 9)
Parcelas: 197
Nulos: centroid         0
all_touched      0
area_weighted    0
dtype: int64


In [30]:
print("=== PROCESANDO ELEVACIÓN — 197 PARCELAS ===")

elevacion_comparacion = extraer_topografia_comparacion(
    ELEV_PATH,
    parcels
)

print("\nShape:", elevacion_comparacion.shape)
print(
    "Parcelas:",
    elevacion_comparacion["ID_POLIGON"].nunique()
)

print(
    "Nulos:",
    elevacion_comparacion[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].isna().sum()
)

=== PROCESANDO ELEVACIÓN — 197 PARCELAS ===

Shape: (197, 9)
Parcelas: 197
Nulos: centroid         0
all_touched      0
area_weighted    0
dtype: int64


In [31]:
print("=== PROCESANDO ELEVACIÓN — 197 PARCELAS ===")

elevacion_comparacion = extraer_topografia_comparacion(
    ELEV_PATH,
    parcels
)

print("\nShape:", elevacion_comparacion.shape)
print(
    "Parcelas:",
    elevacion_comparacion["ID_POLIGON"].nunique()
)

print(
    "Nulos:",
    elevacion_comparacion[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].isna().sum()
)

=== PROCESANDO ELEVACIÓN — 197 PARCELAS ===

Shape: (197, 9)
Parcelas: 197
Nulos: centroid         0
all_touched      0
area_weighted    0
dtype: int64


In [32]:
elevacion_comparacion["error_area_m2"] = (
    elevacion_comparacion[
        "area_intersecciones_m2"
    ]
    -
    elevacion_comparacion[
        "area_parcela_m2"
    ]
).abs()

elevacion_comparacion["error_pesos"] = (
    elevacion_comparacion["suma_pesos"] - 1
).abs()

print("=== INTEGRIDAD GEOMÉTRICA ===")

print(
    "Cobertura mínima:",
    elevacion_comparacion[
        "cobertura_valida"
    ].min()
)

print(
    "Cobertura máxima:",
    elevacion_comparacion[
        "cobertura_valida"
    ].max()
)

print(
    "Máximo error de área (m²):",
    elevacion_comparacion[
        "error_area_m2"
    ].max()
)

print(
    "Máximo error de pesos:",
    elevacion_comparacion[
        "error_pesos"
    ].max()
)

print(
    "Parcelas con cobertura < 1:",
    (
        elevacion_comparacion[
            "cobertura_valida"
        ] < 1 - 1e-10
    ).sum()
)

print(
    "Parcelas con pesos != 1:",
    (
        elevacion_comparacion[
            "error_pesos"
        ] > 1e-10
    ).sum()
)

=== INTEGRIDAD GEOMÉTRICA ===
Cobertura mínima: 0.7593801503373847
Cobertura máxima: 1.0000000000016684
Máximo error de área (m²): 19437.291025034632
Máximo error de pesos: 0.2406198496626153
Parcelas con cobertura < 1: 4
Parcelas con pesos != 1: 4


In [33]:
print("=== DISTRIBUCIÓN DE PÍXELES ===")

print(
    elevacion_comparacion[
        "n_pixeles"
    ].describe()
)

print("\nFrecuencia:")
print(
    elevacion_comparacion[
        "n_pixeles"
    ].value_counts()
    .sort_index()
)

=== DISTRIBUCIÓN DE PÍXELES ===
count    197.000000
mean      11.395939
std        7.577867
min        1.000000
25%        7.000000
50%       10.000000
75%       14.000000
max       61.000000
Name: n_pixeles, dtype: float64

Frecuencia:
n_pixeles
1      1
2      2
3      2
4     10
5     16
6     18
7     12
8     19
9     10
10    11
11    23
12    10
13    11
14     6
15     8
16     4
17     9
18     4
19     5
20     4
22     1
23     1
24     2
26     2
27     2
29     1
37     1
60     1
61     1
Name: count, dtype: int64


In [34]:
problemas_elevacion = elevacion_comparacion[
    elevacion_comparacion["cobertura_valida"] < 1 - 1e-10
].copy()

print(problemas_elevacion[
    [
        "ID_POLIGON",
        "n_pixeles",
        "cobertura_valida",
        "area_intersecciones_m2",
        "area_parcela_m2",
        "centroid",
        "all_touched",
        "area_weighted"
    ]
].sort_values("cobertura_valida"))

    ID_POLIGON  n_pixeles  cobertura_valida  area_intersecciones_m2  \
175    AGC_170         11          0.759380            61342.790304   
178    AGC_164         15          0.950612           111822.470402   
163    AGC_023         11          0.958995            73537.912501   
2      AGC_020         11          0.978321            81059.998963   

     area_parcela_m2  centroid  all_touched  area_weighted  
175     80780.081329    2502.0  2501.272727    1899.382684  
178    117632.076023    2498.0  2497.200000    2373.344149  
163     76682.271506    2500.0  2503.454545    2399.843957  
2       82856.226598    2529.0  2529.090909    2474.100648  


In [35]:
print("Número de parcelas problemáticas:", len(problemas_elevacion))

print("\n=== COBERTURA ===")
print(problemas_elevacion["cobertura_valida"].describe())

print("\n=== ÁREA FALTANTE ===")
problemas_elevacion["area_faltante_m2"] = (
    problemas_elevacion["area_parcela_m2"]
    - problemas_elevacion["area_intersecciones_m2"]
)

print(
    problemas_elevacion[
        ["ID_POLIGON", "area_faltante_m2", "cobertura_valida"]
    ].sort_values("area_faltante_m2", ascending=False)
)

Número de parcelas problemáticas: 4

=== COBERTURA ===
count    4.000000
mean     0.911827
std      0.102291
min      0.759380
25%      0.902804
50%      0.954804
75%      0.963827
max      0.978321
Name: cobertura_valida, dtype: float64

=== ÁREA FALTANTE ===
    ID_POLIGON  area_faltante_m2  cobertura_valida
175    AGC_170      19437.291025          0.759380
178    AGC_164       5809.605621          0.950612
163    AGC_023       3144.359006          0.958995
2      AGC_020       1796.227635          0.978321


In [36]:
id_test = "AGC_170"

fila = parcels[parcels["ID_POLIGON"] == id_test].iloc[0]

parcel_metric = (
    parcels[parcels["ID_POLIGON"] == id_test]
    .to_crs("EPSG:6372")
    .geometry.iloc[0]
)

with rasterio.open(ELEV_PATH) as src:
    print("=== RASTER ===")
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("NoData:", src.nodata)

    print("\n=== PARCELA ===")
    print("ID:", id_test)
    print("Área:", parcel_metric.area, "m²")
    print("Bounds:", parcel_metric.bounds)

    print("\n=== INTERSECCIÓN CON RASTER ===")
    raster_box = box(*src.bounds)
    inter = parcel_metric.intersection(raster_box)

    print("Interseca raster:", parcel_metric.intersects(raster_box))
    print("Área dentro del raster:", inter.area, "m²")
    print("Área fuera del raster:", parcel_metric.area - inter.area, "m²")
    print("Cobertura geométrica:", inter.area / parcel_metric.area)

=== RASTER ===
CRS: EPSG:6372
Bounds: BoundingBox(left=2715840.0, bottom=650400.0, right=3063840.0, top=1053840.0)
NoData: -9999.0

=== PARCELA ===
ID: AGC_170
Área: 80780.08132865428 m²
Bounds: (2863102.643230205, 849731.7745489926, 2863526.441601553, 850160.9329382001)

=== INTERSECCIÓN CON RASTER ===
Interseca raster: True
Área dentro del raster: 80780.08132865427 m²
Área fuera del raster: 1.4551915228366852e-11 m²
Cobertura geométrica: 0.9999999999999998


In [37]:
import math
from rasterio.windows import from_bounds, Window

id_test = "AGC_170"

parcel_metric = (
    parcels[parcels["ID_POLIGON"] == id_test]
    .to_crs("EPSG:6372")
    .geometry.iloc[0]
)

with rasterio.open(ELEV_PATH) as src:

    # Ventana completa que contiene toda la parcela
    raw_window = from_bounds(*parcel_metric.bounds, transform=src.transform)

    col_start = math.floor(raw_window.col_off)
    row_start = math.floor(raw_window.row_off)

    col_end = math.ceil(
        raw_window.col_off + raw_window.width
    )

    row_end = math.ceil(
        raw_window.row_off + raw_window.height
    )

    window = Window(
        col_start,
        row_start,
        col_end - col_start,
        row_end - row_start
    )

    data = src.read(1, window=window)
    transform = src.window_transform(window)

    print("=== VENTANA ===")
    print("Window:", window)
    print("Shape:", data.shape)

    print("\n=== PÍXELES QUE INTERSECTAN LA PARCELA ===")

    resultados = []

    for r in range(data.shape[0]):
        for c in range(data.shape[1]):

            value = data[r, c]

            # Geometría del píxel
            x_left, y_top = transform * (c, r)
            x_right, y_bottom = transform * (c + 1, r + 1)

            pixel = box(
                x_left,
                y_bottom,
                x_right,
                y_top
            )

            if not pixel.intersects(parcel_metric):
                continue

            inter = pixel.intersection(parcel_metric)

            if inter.is_empty:
                continue

            resultados.append({
                "row": row_start + r,
                "col": col_start + c,
                "valor": float(value),
                "es_nodata": bool(
                    value == src.nodata
                ),
                "area_interseccion_m2": inter.area
            })

df_pixeles = pd.DataFrame(resultados)

print(df_pixeles.to_string(index=False))

print("\n=== RESUMEN ===")
print("Píxeles que intersectan:", len(df_pixeles))
print(
    "Píxeles válidos:",
    (~df_pixeles["es_nodata"]).sum()
)
print(
    "Píxeles NoData:",
    df_pixeles["es_nodata"].sum()
)

print(
    "Área total de intersección:",
    df_pixeles["area_interseccion_m2"].sum()
)

print(
    "Área intersección válida:",
    df_pixeles.loc[
        ~df_pixeles["es_nodata"],
        "area_interseccion_m2"
    ].sum()
)

=== VENTANA ===
Window: Window(col_off=1227, row_off=1697, width=4, height=4)
Shape: (4, 4)

=== PÍXELES QUE INTERSECTAN LA PARCELA ===
 row  col   valor  es_nodata  area_interseccion_m2
1697 1227  2500.0      False            686.241524
1697 1228 -9999.0       True           5891.424349
1697 1229  2499.0      False             12.438936
1698 1227  2502.0      False           8090.424254
1698 1228  2500.0      False          14400.000000
1698 1229 -9999.0       True           7515.409834
1698 1230  2499.0      False              0.069305
1699 1227  2503.0      False            722.296739
1699 1228  2502.0      False          11279.982695
1699 1229  2500.0      False          14400.000000
1699 1230 -9999.0       True           6030.456842
1700 1228  2505.0      False            971.578187
1700 1229  2503.0      False           9389.559129
1700 1230  2501.0      False           1390.199535

=== RESUMEN ===
Píxeles que intersectan: 14
Píxeles válidos: 11
Píxeles NoData: 3
Área total de in

In [38]:
print("=== PATRÓN ESPACIAL DE AGC_170 ===")

for r in range(data.shape[0]):
    fila = []

    for c in range(data.shape[1]):
        value = data[r, c]

        if value == -9999:
            fila.append("N")
        else:
            fila.append("V")

    print(" ".join(fila))

=== PATRÓN ESPACIAL DE AGC_170 ===
V N V V
V V N V
V V V N
V V V V


In [39]:
print("\n=== MATRIZ DE ELEVACIÓN ===")
print(data)


=== MATRIZ DE ELEVACIÓN ===
[[ 2500. -9999.  2499.  2498.]
 [ 2502.  2500. -9999.  2499.]
 [ 2503.  2502.  2500. -9999.]
 [ 2506.  2505.  2503.  2501.]]


In [40]:
ids_test = ["AGC_164", "AGC_023", "AGC_020"]

for id_test in ids_test:

    parcel_metric = (
        parcels[parcels["ID_POLIGON"] == id_test]
        .to_crs("EPSG:6372")
        .geometry.iloc[0]
    )

    with rasterio.open(ELEV_PATH) as src:

        raw_window = from_bounds(
            *parcel_metric.bounds,
            transform=src.transform
        )

        col_start = math.floor(raw_window.col_off)
        row_start = math.floor(raw_window.row_off)

        col_end = math.ceil(
            raw_window.col_off + raw_window.width
        )

        row_end = math.ceil(
            raw_window.row_off + raw_window.height
        )

        window = Window(
            col_start,
            row_start,
            col_end - col_start,
            row_end - row_start
        )

        data_test = src.read(1, window=window)

        print("\n" + "=" * 50)
        print(f"PARCELA: {id_test}")
        print("=" * 50)

        print("Window:", window)
        print("Shape:", data_test.shape)

        print("\nPatrón:")
        for r in range(data_test.shape[0]):
            fila = []

            for c in range(data_test.shape[1]):
                if data_test[r, c] == src.nodata:
                    fila.append("N")
                else:
                    fila.append("V")

            print(" ".join(fila))

        print("\nMatriz:")
        print(data_test)

        print(
            "NoData:",
            np.sum(data_test == src.nodata)
        )


PARCELA: AGC_164
Window: Window(col_off=1230, row_off=1696, width=4, height=6)
Shape: (6, 4)

Patrón:
V V V V
V V V V
V V V V
N V V V
V V V V
V V V V

Matriz:
[[ 2497.  2495.  2493.  2493.]
 [ 2498.  2496.  2494.  2493.]
 [ 2499.  2498.  2496.  2494.]
 [-9999.  2499.  2498.  2496.]
 [ 2501.  2500.  2499.  2497.]
 [ 2504.  2500.  2500.  2498.]]
NoData: 1

PARCELA: AGC_023
Window: Window(col_off=1229, row_off=1699, width=4, height=5)
Shape: (5, 4)

Patrón:
V N V V
V V V V
V V V V
V V V N
V V V V

Matriz:
[[ 2500. -9999.  2499.  2498.]
 [ 2503.  2501.  2500.  2499.]
 [ 2507.  2504.  2500.  2500.]
 [ 2509.  2506.  2503. -9999.]
 [ 2510.  2509.  2505.  2501.]]
NoData: 2

PARCELA: AGC_020
Window: Window(col_off=1174, row_off=1681, width=4, height=4)
Shape: (4, 4)

Patrón:
V V V N
V V V N
V V V V
V V V V

Matriz:
[[ 2530.  2530.  2529. -9999.]
 [ 2531.  2530.  2528. -9999.]
 [ 2530.  2529.  2528.  2527.]
 [ 2529.  2528.  2528.  2530.]]
NoData: 2


In [41]:
with rasterio.open(ELEV_PATH) as src:

    row_nodata = 1697
    col_nodata = 1228

    ventana_vecina = src.read(
        1,
        window=Window(
            col_nodata - 1,
            row_nodata - 1,
            3,
            3
        )
    )

print(ventana_vecina)

[[-9999.  2499.  2498.]
 [ 2500. -9999.  2499.]
 [ 2502.  2500. -9999.]]


In [42]:
ids_problematicos = [
    "AGC_170",
    "AGC_164",
    "AGC_023",
    "AGC_020"
]

resultados_nodata = []

with rasterio.open(ELEV_PATH) as src:

    nodata = src.nodata

    for id_test in ids_problematicos:

        parcel_metric = (
            parcels[parcels["ID_POLIGON"] == id_test]
            .to_crs("EPSG:6372")
            .geometry.iloc[0]
        )

        raw_window = from_bounds(
            *parcel_metric.bounds,
            transform=src.transform
        )

        col_start = math.floor(raw_window.col_off)
        row_start = math.floor(raw_window.row_off)

        col_end = math.ceil(
            raw_window.col_off + raw_window.width
        )

        row_end = math.ceil(
            raw_window.row_off + raw_window.height
        )

        window = Window(
            col_start,
            row_start,
            col_end - col_start,
            row_end - row_start
        )

        data_local = src.read(1, window=window)

        for r in range(data_local.shape[0]):
            for c in range(data_local.shape[1]):

                if data_local[r, c] != nodata:
                    continue

                row_global = row_start + r
                col_global = col_start + c

                # Vecindario 3x3
                r0 = max(0, row_global - 1)
                r1 = min(src.height, row_global + 2)

                c0 = max(0, col_global - 1)
                c1 = min(src.width, col_global + 2)

                vecinos = src.read(
                    1,
                    window=Window(
                        c0,
                        r0,
                        c1 - c0,
                        r1 - r0
                    )
                )

                validos = vecinos[
                    vecinos != nodata
                ]

                resultados_nodata.append({
                    "ID_POLIGON": id_test,
                    "row": row_global,
                    "col": col_global,
                    "n_vecinos_validos": len(validos),
                    "min_vecinos": validos.min() if len(validos) else np.nan,
                    "max_vecinos": validos.max() if len(validos) else np.nan,
                    "media_vecinos": validos.mean() if len(validos) else np.nan,
                    "rango_vecinos": (
                        validos.max() - validos.min()
                        if len(validos)
                        else np.nan
                    )
                })

df_nodata = pd.DataFrame(resultados_nodata)

print(df_nodata.to_string(index=False))

ID_POLIGON  row  col  n_vecinos_validos  min_vecinos  max_vecinos  media_vecinos  rango_vecinos
   AGC_170 1697 1228                  6       2498.0       2502.0    2499.666748            4.0
   AGC_170 1698 1229                  6       2498.0       2502.0    2499.666748            4.0
   AGC_170 1699 1230                  7       2498.0       2503.0    2500.000000            5.0
   AGC_164 1699 1230                  7       2498.0       2503.0    2500.000000            5.0
   AGC_023 1699 1230                  7       2498.0       2503.0    2500.000000            5.0
   AGC_023 1702 1232                  8       2498.0       2505.0    2500.625000            7.0
   AGC_020 1681 1177                  6       2525.0       2530.0    2527.666748            5.0
   AGC_020 1682 1177                  7       2526.0       2533.0    2528.428467            7.0


In [43]:
import numpy as np
import pandas as pd
import rasterio

with rasterio.open(ELEV_PATH) as src:

    elevation = src.read(1)
    nodata = src.nodata

print("Shape raster:", elevation.shape)
print("NoData:", nodata)

# ---------------------------------------------------------
# 1. Identificar píxeles válidos
# ---------------------------------------------------------

valid_mask = (
    (elevation != nodata)
    & np.isfinite(elevation)
)

print("Píxeles válidos:", valid_mask.sum())
print("Píxeles NoData:", (~valid_mask).sum())

# ---------------------------------------------------------
# 2. Buscar píxeles válidos cuyos 8 vecinos
#    también sean válidos
# ---------------------------------------------------------

candidatos = []

rows, cols = elevation.shape

for r in range(1, rows - 1):
    for c in range(1, cols - 1):

        valor_real = elevation[r, c]

        if not valid_mask[r, c]:
            continue

        vecinos = elevation[
            r - 1:r + 2,
            c - 1:c + 2
        ]

        vecinos_mask = valid_mask[
            r - 1:r + 2,
            c - 1:c + 2
        ]

        # Los 8 vecinos alrededor del píxel central
        vecinos_validos = vecinos[
            vecinos_mask
        ]

        # Incluye el propio píxel central,
        # por lo que deben existir 9 valores válidos
        if len(vecinos_validos) != 9:
            continue

        # Quitamos el píxel central
        vecinos_sin_centro = np.delete(
            vecinos_validos,
            4
        )

        estimacion = vecinos_sin_centro.mean()

        error_abs = abs(
            valor_real - estimacion
        )

        error_rel = (
            error_abs / abs(valor_real)
        ) * 100

        candidatos.append({
            "row": r,
            "col": c,
            "real_m": float(valor_real),
            "estimado_m": float(estimacion),
            "error_abs_m": float(error_abs),
            "error_rel_pct": float(error_rel)
        })

df_validacion_imputacion = pd.DataFrame(candidatos)

print("\n=== VALIDACIÓN DE IMPUTACIÓN ===")
print("Candidatos:", len(df_validacion_imputacion))

print("\nPrimeros resultados:")
print(df_validacion_imputacion.head(10))

Shape raster: (3362, 2900)
NoData: -9999.0
Píxeles válidos: 4081208
Píxeles NoData: 5668592

=== VALIDACIÓN DE IMPUTACIÓN ===
Candidatos: 4047765

Primeros resultados:
   row   col  real_m  estimado_m  error_abs_m  error_rel_pct
0   56  1224    53.0      50.750        2.250       4.245283
1   56  1225    53.0      51.000        2.000       3.773585
2   56  1226    51.0      50.375        0.625       1.225490
3   57  1224    52.0      51.625        0.375       0.721154
4   57  1225    53.0      52.375        0.625       1.179245
5   57  1226    52.0      51.625        0.375       0.721154
6   58  1224    52.0      51.750        0.250       0.480769
7   58  1225    53.0      52.875        0.125       0.235849
8   58  1226    53.0      52.125        0.875       1.650943
9   59  1224    53.0      52.625        0.375       0.707547


In [44]:
print("=== ERROR ABSOLUTO DE RECONSTRUCCIÓN ===")

print(
    df_validacion_imputacion["error_abs_m"].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\n=== ERROR RELATIVO (%) ===")

print(
    df_validacion_imputacion["error_rel_pct"].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\n=== ERRORES MÁXIMOS ===")

print(
    df_validacion_imputacion[
        [
            "row",
            "col",
            "real_m",
            "estimado_m",
            "error_abs_m",
            "error_rel_pct"
        ]
    ]
    .sort_values(
        "error_abs_m",
        ascending=False
    )
    .head(20)
)

=== ERROR ABSOLUTO DE RECONSTRUCCIÓN ===
count    4.047765e+06
mean     8.954669e+00
std      1.137915e+01
min      0.000000e+00
50%      4.500000e+00
75%      1.300000e+01
90%      2.412500e+01
95%      3.200000e+01
99%      4.925000e+01
max      3.040000e+02
Name: error_abs_m, dtype: float64

=== ERROR RELATIVO (%) ===
count    4.047765e+06
mean     8.332108e-01
std      4.729337e+00
min      0.000000e+00
50%      2.694421e-01
75%      9.060651e-01
90%      2.004999e+00
95%      3.144934e+00
99%      7.426598e+00
max      1.865625e+03
Name: error_rel_pct, dtype: float64

=== ERRORES MÁXIMOS ===
          row   col  real_m  estimado_m  error_abs_m  error_rel_pct
2722864  2259  2326  3051.0    3355.000      304.000       9.963946
3821782  2997  1351   516.0     785.375      269.375      52.204460
3827336  3001  1328   521.0     789.125      268.125      51.463531
3820366  2996  1351   516.0     783.625      267.625      51.865311
3825978  3000  1347   516.0     782.250      266.250    

In [45]:
# =========================================================
# COMPARACIÓN: ELEVACIÓN OBSERVADA VS IMPUTADA
# SOLO 4 PARCELAS PROBLEMÁTICAS
# =========================================================

import math
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds, Window
from shapely.geometry import box


ids_problematicos = [
    "AGC_170",
    "AGC_164",
    "AGC_023",
    "AGC_020"
]

resultados_comparacion = []


with rasterio.open(ELEV_PATH) as src:

    nodata = src.nodata

    # -----------------------------------------------------
    # Reproyectar parcelas al CRS del raster
    # -----------------------------------------------------

    parcels_metric = parcels.to_crs(src.crs)

    for id_test in ids_problematicos:

        parcel = parcels_metric[
            parcels_metric["ID_POLIGON"] == id_test
        ].iloc[0]

        geom = parcel.geometry

        area_parcela = geom.area

        # -------------------------------------------------
        # Ventana completa que contiene la parcela
        # -------------------------------------------------

        raw_window = from_bounds(
            *geom.bounds,
            transform=src.transform
        )

        col_start = math.floor(raw_window.col_off)
        row_start = math.floor(raw_window.row_off)

        col_end = math.ceil(
            raw_window.col_off + raw_window.width
        )

        row_end = math.ceil(
            raw_window.row_off + raw_window.height
        )

        window = Window(
            col_start,
            row_start,
            col_end - col_start,
            row_end - row_start
        )

        data = src.read(
            1,
            window=window
        )

        transform = src.window_transform(window)

        # -------------------------------------------------
        # Recorrer píxeles
        # -------------------------------------------------

        registros_pixeles = []

        for r in range(data.shape[0]):
            for c in range(data.shape[1]):

                value = data[r, c]

                # Geometría del píxel
                x_left, y_top = transform * (c, r)
                x_right, y_bottom = transform * (c + 1, r + 1)

                pixel = box(
                    x_left,
                    y_bottom,
                    x_right,
                    y_top
                )

                if not pixel.intersects(geom):
                    continue

                inter = pixel.intersection(geom)

                if inter.is_empty or inter.area <= 0:
                    continue

                es_nodata = (
                    value == nodata
                    or not np.isfinite(value)
                )

                # -------------------------------------------------
                # Valor imputado si es NoData
                # -------------------------------------------------

                valor_imputado = np.nan

                if es_nodata:

                    row_global = row_start + r
                    col_global = col_start + c

                    # Vecindario 3x3 dentro del raster
                    r0 = max(0, row_global - 1)
                    r1 = min(src.height, row_global + 2)

                    c0 = max(0, col_global - 1)
                    c1 = min(src.width, col_global + 2)

                    vecinos = src.read(
                        1,
                        window=Window(
                            c0,
                            r0,
                            c1 - c0,
                            r1 - r0
                        )
                    )

                    vecinos_validos = vecinos[
                        (vecinos != nodata)
                        & np.isfinite(vecinos)
                    ]

                    if len(vecinos_validos) > 0:
                        valor_imputado = (
                            vecinos_validos.mean()
                        )

                registros_pixeles.append({
                    "valor": (
                        np.nan
                        if es_nodata
                        else float(value)
                    ),
                    "valor_imputado": (
                        float(value)
                        if not es_nodata
                        else valor_imputado
                    ),
                    "es_nodata": es_nodata,
                    "area_interseccion_m2": inter.area
                })

        df_pix = pd.DataFrame(registros_pixeles)

        # -----------------------------------------------------
        # Área válida
        # -----------------------------------------------------

        area_valida = df_pix.loc[
            ~df_pix["es_nodata"],
            "area_interseccion_m2"
        ].sum()

        cobertura = area_valida / area_parcela

        # -----------------------------------------------------
        # Elevación observada
        # -----------------------------------------------------

        pesos_validos = (
            df_pix.loc[
                ~df_pix["es_nodata"],
                "area_interseccion_m2"
            ] / area_parcela
        )

        valores_validos = df_pix.loc[
            ~df_pix["es_nodata"],
            "valor"
        ]

        elevacion_valida = (
            valores_validos * pesos_validos
        ).sum()

        # -----------------------------------------------------
        # Elevación imputada
        # -----------------------------------------------------

        valores_imputados = df_pix[
            "valor_imputado"
        ]

        pesos_totales = (
            df_pix["area_interseccion_m2"]
            / area_parcela
        )

        elevacion_imputada = (
            valores_imputados * pesos_totales
        ).sum()

        # -----------------------------------------------------
        # Resultado
        # -----------------------------------------------------

        resultados_comparacion.append({
            "ID_POLIGON": id_test,
            "cobertura_valida": cobertura,
            "elevacion_valida_m": elevacion_valida,
            "elevacion_imputada_m": elevacion_imputada,
            "diferencia_m": (
                elevacion_imputada
                - elevacion_valida
            ),
            "n_pixeles": len(df_pix),
            "n_nodata": int(
                df_pix["es_nodata"].sum()
            )
        })


comparacion_elevacion = pd.DataFrame(
    resultados_comparacion
)

print("=== COMPARACIÓN ELEVACIÓN ===")

print(
    comparacion_elevacion.to_string(
        index=False
    )
)

=== COMPARACIÓN ELEVACIÓN ===
ID_POLIGON  cobertura_valida  elevacion_valida_m  elevacion_imputada_m  diferencia_m  n_pixeles  n_nodata
   AGC_170          0.759380         1899.382684           2500.877000    601.494315         14         3
   AGC_164          0.950612         2373.344149           2496.813993    123.469844         16         1
   AGC_023          0.958995         2399.843957           2502.376589    102.532633         13         2
   AGC_020          0.978321         2474.100648           2528.909696     54.809048         13         2


In [46]:
print("\n=== DIFERENCIA ABSOLUTA ===")

comparacion_elevacion[
    "diferencia_m"
] = comparacion_elevacion[
    "diferencia_m"
].abs()

print(
    comparacion_elevacion[
        [
            "ID_POLIGON",
            "cobertura_valida",
            "elevacion_valida_m",
            "elevacion_imputada_m",
            "diferencia_m",
            "n_nodata"
        ]
    ]
)


=== DIFERENCIA ABSOLUTA ===
  ID_POLIGON  cobertura_valida  elevacion_valida_m  elevacion_imputada_m  \
0    AGC_170          0.759380         1899.382684           2500.877000   
1    AGC_164          0.950612         2373.344149           2496.813993   
2    AGC_023          0.958995         2399.843957           2502.376589   
3    AGC_020          0.978321         2474.100648           2528.909696   

   diferencia_m  n_nodata  
0    601.494315         3  
1    123.469844         1  
2    102.532633         2  
3     54.809048         2  


In [47]:
# =========================================================
# CORRECCIÓN:
# ELEVACIÓN OBSERVADA NORMALIZADA POR ÁREA VÁLIDA
# VS ELEVACIÓN IMPUTADA
# =========================================================

comparacion_corregida = comparacion_elevacion.copy()

# Cobertura válida ya calculada
# La elevación observada anterior estaba ponderada
# contra el área total, por lo que corregimos:

comparacion_corregida[
    "elevacion_observada_corregida_m"
] = (
    comparacion_corregida["elevacion_valida_m"]
    / comparacion_corregida["cobertura_valida"]
)

comparacion_corregida[
    "diferencia_real_m"
] = (
    comparacion_corregida[
        "elevacion_imputada_m"
    ]
    -
    comparacion_corregida[
        "elevacion_observada_corregida_m"
    ]
).abs()

print(
    comparacion_corregida[
        [
            "ID_POLIGON",
            "cobertura_valida",
            "elevacion_observada_corregida_m",
            "elevacion_imputada_m",
            "diferencia_real_m",
            "n_nodata"
        ]
    ].to_string(index=False)
)

ID_POLIGON  cobertura_valida  elevacion_observada_corregida_m  elevacion_imputada_m  diferencia_real_m  n_nodata
   AGC_170          0.759380                      2501.227723           2500.877000           0.350723         3
   AGC_164          0.950612                      2496.648468           2496.813993           0.165525         1
   AGC_023          0.958995                      2502.457298           2502.376589           0.080709         2
   AGC_020          0.978321                      2528.924828           2528.909696           0.015132         2


In [48]:
def extraer_topografia_comparacion(raster_path, parcels_gdf):
    """
    Extrae estadísticas de un raster topográfico por parcela.

    Métodos:
    - centroid: valor del píxel que contiene el centroide.
    - all_touched: media simple de píxeles válidos que intersectan la parcela.
    - area_weighted: media ponderada por área de los píxeles válidos.

    Importante:
    - Los píxeles NoData NO se imputan.
    - area_weighted se normaliza usando únicamente el área válida.
    - cobertura_valida indica qué proporción de la parcela tiene datos válidos.
    - Si la cobertura es parcial, se conserva esa información explícitamente.
    """

    import math
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import rasterio
    from shapely.geometry import box
    from rasterio.windows import from_bounds, Window

    resultados = []

    with rasterio.open(raster_path) as src:

        # -------------------------------------------------
        # Validaciones del raster
        # -------------------------------------------------

        if src.crs is None:
            raise ValueError("El raster no tiene CRS.")

        if not src.crs.is_projected:
            raise ValueError(
                "El raster debe estar en un CRS proyectado "
                "para calcular áreas correctamente."
            )

        nodata = src.nodata

        # -------------------------------------------------
        # Transformar parcelas al CRS del raster
        # -------------------------------------------------

        parcels_metric = parcels_gdf.to_crs(src.crs)

        for _, parcel in parcels_metric.iterrows():

            parcel_id = parcel["ID_POLIGON"]
            geom = parcel.geometry

            if geom is None or geom.is_empty:
                raise ValueError(
                    f"Geometría vacía en {parcel_id}"
                )

            if not geom.is_valid:
                raise ValueError(
                    f"Geometría inválida en {parcel_id}"
                )

            area_parcela = geom.area

            if area_parcela <= 0:
                raise ValueError(
                    f"Área no válida en {parcel_id}"
                )

            # -------------------------------------------------
            # Crear ventana completa alrededor de la parcela
            # -------------------------------------------------

            raw_window = from_bounds(
                *geom.bounds,
                transform=src.transform
            )

            col_start = math.floor(
                raw_window.col_off
            )

            row_start = math.floor(
                raw_window.row_off
            )

            col_end = math.ceil(
                raw_window.col_off
                + raw_window.width
            )

            row_end = math.ceil(
                raw_window.row_off
                + raw_window.height
            )

            # -------------------------------------------------
            # Limitar ventana al raster
            # -------------------------------------------------

            col_start = max(0, col_start)
            row_start = max(0, row_start)

            col_end = min(src.width, col_end)
            row_end = min(src.height, row_end)

            width = col_end - col_start
            height = row_end - row_start

            if width <= 0 or height <= 0:
                raise ValueError(
                    f"La parcela {parcel_id} no tiene "
                    "ventana válida dentro del raster."
                )

            window = Window(
                col_start,
                row_start,
                width,
                height
            )

            data = src.read(
                1,
                window=window
            )

            transform = src.window_transform(window)

            # -------------------------------------------------
            # Información de píxeles
            # -------------------------------------------------

            registros = []

            for r in range(data.shape[0]):
                for c in range(data.shape[1]):

                    value = data[r, c]

                    # Geometría del píxel
                    x_left, y_top = transform * (c, r)
                    x_right, y_bottom = transform * (
                        c + 1,
                        r + 1
                    )

                    pixel = box(
                        x_left,
                        y_bottom,
                        x_right,
                        y_top
                    )

                    if not pixel.intersects(geom):
                        continue

                    inter = pixel.intersection(geom)

                    if inter.is_empty or inter.area <= 0:
                        continue

                    # -------------------------------------------------
                    # Validación del valor
                    # -------------------------------------------------

                    es_nodata = (
                        value == nodata
                        or not np.isfinite(value)
                    )

                    registros.append({
                        "row": row_start + r,
                        "col": col_start + c,
                        "valor": (
                            float(value)
                            if not es_nodata
                            else np.nan
                        ),
                        "es_nodata": es_nodata,
                        "area_interseccion_m2": inter.area
                    })

            df_pix = pd.DataFrame(registros)

            if df_pix.empty:
                raise ValueError(
                    f"No se encontraron píxeles "
                    f"para {parcel_id}"
                )

            # -------------------------------------------------
            # Píxeles válidos
            # -------------------------------------------------

            validos = df_pix[
                ~df_pix["es_nodata"]
            ].copy()

            if validos.empty:
                raise ValueError(
                    f"La parcela {parcel_id} no tiene "
                    "ningún píxel válido."
                )

            area_valida = (
                validos[
                    "area_interseccion_m2"
                ].sum()
            )

            cobertura_valida = (
                area_valida / area_parcela
            )

            # -------------------------------------------------
            # 1. CENTROIDE
            # -------------------------------------------------

            centroid_point = geom.centroid

            centroid_value = np.nan

            for _, p in validos.iterrows():

                x_left, y_top = transform * (
                    p["col"] - col_start,
                    p["row"] - row_start
                )

                x_right, y_bottom = transform * (
                    p["col"] - col_start + 1,
                    p["row"] - row_start + 1
                )

                pixel_geom = box(
                    x_left,
                    y_bottom,
                    x_right,
                    y_top
                )

                if pixel_geom.covers(
                    centroid_point
                ):
                    centroid_value = p["valor"]
                    break

            # -------------------------------------------------
            # 2. ALL TOUCHED
            # -------------------------------------------------

            all_touched_value = (
                validos["valor"].mean()
            )

            # -------------------------------------------------
            # 3. AREA WEIGHTED CORRECTO
            # -------------------------------------------------

            validos["peso_area_valida"] = (
                validos[
                    "area_interseccion_m2"
                ] / area_valida
            )

            area_weighted_value = (
                (
                    validos["valor"]
                    * validos["peso_area_valida"]
                ).sum()
            )

            suma_pesos = (
                validos["peso_area_valida"]
                .sum()
            )

            # -------------------------------------------------
            # Resultado
            # -------------------------------------------------

            resultados.append({
                "ID_POLIGON": parcel_id,
                "centroid": centroid_value,
                "all_touched": all_touched_value,
                "area_weighted": area_weighted_value,
                "n_pixeles": len(df_pix),
                "n_pixeles_validos": len(validos),
                "n_pixeles_nodata": int(
                    df_pix["es_nodata"].sum()
                ),
                "suma_pesos": suma_pesos,
                "area_intersecciones_m2": area_valida,
                "area_parcela_m2": area_parcela,
                "cobertura_valida": cobertura_valida
            })

    return pd.DataFrame(resultados)

In [49]:
elevacion_comparacion = extraer_topografia_comparacion(
    ELEV_PATH,
    parcels
)

print("Shape:", elevacion_comparacion.shape)

print(
    elevacion_comparacion[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].isna().sum()
)

Shape: (197, 11)
centroid         0
all_touched      0
area_weighted    0
dtype: int64


In [50]:
elevacion_comparacion["error_area_m2"] = (
    elevacion_comparacion[
        "area_intersecciones_m2"
    ]
    -
    elevacion_comparacion[
        "area_parcela_m2"
    ]
).abs()

elevacion_comparacion["error_pesos"] = (
    elevacion_comparacion[
        "suma_pesos"
    ] - 1
).abs()

print("=== INTEGRIDAD GEOMÉTRICA ===")

print(
    "Cobertura mínima:",
    elevacion_comparacion[
        "cobertura_valida"
    ].min()
)

print(
    "Cobertura máxima:",
    elevacion_comparacion[
        "cobertura_valida"
    ].max()
)

print(
    "Máximo error de área:",
    elevacion_comparacion[
        "error_area_m2"
    ].max(),
    "m²"
)

print(
    "Máximo error de pesos:",
    elevacion_comparacion[
        "error_pesos"
    ].max()
)

print(
    "Parcelas con cobertura < 1:",
    (
        elevacion_comparacion[
            "cobertura_valida"
        ] < 1 - 1e-10
    ).sum()
)

print(
    "Pesos con error > 1e-10:",
    (
        elevacion_comparacion[
            "error_pesos"
        ] > 1e-10
    ).sum()
)

=== INTEGRIDAD GEOMÉTRICA ===
Cobertura mínima: 0.7593801503373847
Cobertura máxima: 1.0000000000016684
Máximo error de área: 19437.291025034632 m²
Máximo error de pesos: 4.440892098500626e-16
Parcelas con cobertura < 1: 4
Pesos con error > 1e-10: 0


In [51]:
print(
    elevacion_comparacion[
        elevacion_comparacion[
            "cobertura_valida"
        ] < 1 - 1e-10
    ][
        [
            "ID_POLIGON",
            "n_pixeles",
            "n_pixeles_validos",
            "n_pixeles_nodata",
            "cobertura_valida",
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].sort_values(
        "cobertura_valida"
    ).to_string(index=False)
)

ID_POLIGON  n_pixeles  n_pixeles_validos  n_pixeles_nodata  cobertura_valida  centroid  all_touched  area_weighted
   AGC_170         14                 11                 3          0.759380    2502.0  2501.272727    2501.227723
   AGC_164         16                 15                 1          0.950612    2498.0  2497.200000    2496.648468
   AGC_023         13                 11                 2          0.958995    2500.0  2503.454545    2502.457298
   AGC_020         13                 11                 2          0.978321    2529.0  2529.090909    2528.924828


In [52]:
print("\n=== DISTRIBUCIÓN DE ELEVACIÓN ===")

print(
    elevacion_comparacion[
        "area_weighted"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


=== DISTRIBUCIÓN DE ELEVACIÓN ===
count     197.000000
mean     2667.102482
std        92.283855
min      2496.648468
1%       2502.408115
5%       2532.002909
25%      2611.351962
50%      2638.902931
75%      2726.683106
95%      2880.829192
99%      2901.399907
max      2906.039520
Name: area_weighted, dtype: float64


In [53]:
elevacion_comparacion["dif_centroid_area_m"] = (
    elevacion_comparacion["centroid"]
    - elevacion_comparacion["area_weighted"]
).abs()

print("=== CENTROIDE VS ÁREA PONDERADA ===")

print(
    elevacion_comparacion[
        "dif_centroid_area_m"
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\n=== MAYORES DIFERENCIAS ===")

print(
    elevacion_comparacion[
        [
            "ID_POLIGON",
            "centroid",
            "area_weighted",
            "dif_centroid_area_m"
        ]
    ]
    .sort_values(
        "dif_centroid_area_m",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)

=== CENTROIDE VS ÁREA PONDERADA ===
count    197.000000
mean       1.919083
std        2.274996
min        0.000000
50%        1.119054
75%        2.383453
90%        4.720183
95%        6.060935
99%       11.583054
max       12.907884
Name: dif_centroid_area_m, dtype: float64

=== MAYORES DIFERENCIAS ===
ID_POLIGON  centroid  area_weighted  dif_centroid_area_m
   AGC_135    2886.0    2873.092116            12.907884
   AGC_122    2727.0    2715.200862            11.799138
   AGC_119    2630.0    2641.574050            11.574050
   AGC_009    2630.0    2640.888224            10.888224
   AGC_160    2734.0    2743.386441             9.386441
   AGC_181    2649.0    2640.594880             8.405120
   AGC_070    2748.0    2739.829249             8.170751
   AGC_061    2793.0    2785.134727             7.865273
   AGC_051    2742.0    2748.939795             6.939795
   AGC_155    2747.0    2753.303392             6.303392


In [54]:
import numpy as np
import pandas as pd
import rasterio

print("=== AUDITORÍA DEL RASTER DE PENDIENTE ===")

with rasterio.open(SLOPE_PATH) as src:

    print("\n--- METADATA ---")
    print("CRS:", src.crs)
    print("Resolución:", src.res)
    print("Dimensiones:", src.width, "x", src.height)
    print("Bandas:", src.count)
    print("NoData:", src.nodata)
    print("dtype:", src.dtypes[0])
    print("Driver:", src.driver)

    print("\n--- BOUNDS ---")
    print(src.bounds)

    print("\n--- TRANSFORM ---")
    print(src.transform)

    print("\n--- TAGS ---")
    print(src.tags())

    # Leer raster
    slope = src.read(1)

    nodata = src.nodata

    valid_mask = (
        (slope != nodata)
        & np.isfinite(slope)
    )

    valid = slope[valid_mask]

    print("\n--- VALORES ---")
    print("Píxeles totales:", slope.size)
    print("Píxeles válidos:", valid.size)
    print("Píxeles NoData:", (~valid_mask).sum())

    print("\n--- ESTADÍSTICAS ---")
    print("Mínimo:", valid.min())
    print("Máximo:", valid.max())
    print("Media:", valid.mean())
    print("Mediana:", np.median(valid))

    print("\n--- PERCENTILES ---")

    percentiles = np.percentile(
        valid,
        [1, 5, 25, 50, 75, 95, 99]
    )

    for p, value in zip(
        [1, 5, 25, 50, 75, 95, 99],
        percentiles
    ):
        print(f"P{p}: {value:.6f}")

    print("\n--- VALIDACIONES BÁSICAS ---")
    print(
        "Valores negativos:",
        (valid < 0).sum()
    )

    print(
        "Valores > 90°:",
        (valid > 90).sum()
    )

=== AUDITORÍA DEL RASTER DE PENDIENTE ===

--- METADATA ---
CRS: EPSG:6372
Resolución: (120.0, 120.0)
Dimensiones: 2900 x 3362
Bandas: 1
NoData: -9999.0
dtype: float32
Driver: GTiff

--- BOUNDS ---
BoundingBox(left=2715840.0, bottom=650400.0, right=3063840.0, top=1053840.0)

--- TRANSFORM ---
| 120.00, 0.00, 2715840.00|
| 0.00,-120.00, 1053840.00|
| 0.00, 0.00, 1.00|

--- TAGS ---
{'fuente': 'Derivada de Elevacion_INEGI_CEM4_120m.tif', 'metodo': 'Horn 3x3', 'unidad': 'grados', 'AREA_OR_POINT': 'Area'}

--- VALORES ---
Píxeles totales: 9749800
Píxeles válidos: 4047765
Píxeles NoData: 5702035

--- ESTADÍSTICAS ---
Mínimo: 0.0
Máximo: 70.63786
Media: 10.171154
Mediana: 7.143111

--- PERCENTILES ---
P1: 0.119366
P5: 0.477454
P25: 2.565351
P50: 7.143111
P75: 15.501950
P95: 29.337757
P99: 38.499443

--- VALIDACIONES BÁSICAS ---
Valores negativos: 0
Valores > 90°: 0


In [55]:
pendiente_comparacion = extraer_topografia_comparacion(
    SLOPE_PATH,
    parcels
)

print("=== PROCESANDO PENDIENTE — 197 PARCELAS ===")

print("Shape:", pendiente_comparacion.shape)

print("\nNulos:")

print(
    pendiente_comparacion[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].isna().sum()
)

=== PROCESANDO PENDIENTE — 197 PARCELAS ===
Shape: (197, 11)

Nulos:
centroid         3
all_touched      0
area_weighted    0
dtype: int64


In [56]:
pendiente_comparacion["error_pesos"] = (
    pendiente_comparacion[
        "suma_pesos"
    ] - 1
).abs()

print("\n=== INTEGRIDAD ===")

print(
    "Cobertura mínima:",
    pendiente_comparacion[
        "cobertura_valida"
    ].min()
)

print(
    "Cobertura máxima:",
    pendiente_comparacion[
        "cobertura_valida"
    ].max()
)

print(
    "Máximo error de pesos:",
    pendiente_comparacion[
        "error_pesos"
    ].max()
)

print(
    "Parcelas con cobertura < 1:",
    (
        pendiente_comparacion[
            "cobertura_valida"
        ] < 1 - 1e-10
    ).sum()
)

print(
    "Pesos con error > 1e-10:",
    (
        pendiente_comparacion[
            "error_pesos"
        ] > 1e-10
    ).sum()
)


=== INTEGRIDAD ===
Cobertura mínima: 0.020968967816181197
Cobertura máxima: 1.0000000000016684
Máximo error de pesos: 4.440892098500626e-16
Parcelas con cobertura < 1: 5
Pesos con error > 1e-10: 0


In [57]:
print("\n=== DISTRIBUCIÓN DE PÍXELES ===")

print(
    pendiente_comparacion[
        "n_pixeles"
    ].describe()
)

print("\nFrecuencia:")

print(
    pendiente_comparacion[
        "n_pixeles"
    ]
    .value_counts()
    .sort_index()
)


=== DISTRIBUCIÓN DE PÍXELES ===
count    197.000000
mean      11.436548
std        7.584375
min        1.000000
25%        7.000000
50%       10.000000
75%       14.000000
max       61.000000
Name: n_pixeles, dtype: float64

Frecuencia:
n_pixeles
1      1
2      2
3      2
4     10
5     16
6     18
7     12
8     19
9     10
10    11
11    20
12    10
13    13
14     7
15     7
16     5
17     9
18     4
19     5
20     4
22     1
23     1
24     2
26     2
27     2
29     1
37     1
60     1
61     1
Name: count, dtype: int64


In [58]:
problemas_pendiente = pendiente_comparacion[
    pendiente_comparacion["cobertura_valida"] < 1 - 1e-10
].copy()

print(
    problemas_pendiente[
        [
            "ID_POLIGON",
            "n_pixeles",
            "n_pixeles_validos",
            "n_pixeles_nodata",
            "cobertura_valida",
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ]
    .sort_values("cobertura_valida")
    .to_string(index=False)
)

ID_POLIGON  n_pixeles  n_pixeles_validos  n_pixeles_nodata  cobertura_valida  centroid  all_touched  area_weighted
   AGC_170         14                  2                12          0.020969       NaN     1.363419       1.391080
   AGC_023         13                  4                 9          0.286353       NaN     1.662428       1.766181
   AGC_164         16                  8                 8          0.552699       NaN     0.827955       0.903925
   AGC_020         13                  8                 5          0.652503  0.596809     0.511222       0.561396
   AGC_044          4                  3                 1          0.980256  0.304322     0.430841       0.348862


In [59]:
print("\n=== ÁREA SIN DATOS ===")

problemas_pendiente["area_faltante_m2"] = (
    problemas_pendiente["area_parcela_m2"]
    -
    problemas_pendiente["area_intersecciones_m2"]
)

print(
    problemas_pendiente[
        [
            "ID_POLIGON",
            "area_parcela_m2",
            "area_faltante_m2",
            "cobertura_valida",
            "n_pixeles_nodata"
        ]
    ]
    .sort_values(
        "cobertura_valida"
    )
    .to_string(index=False)
)


=== ÁREA SIN DATOS ===
ID_POLIGON  area_parcela_m2  area_faltante_m2  cobertura_valida  n_pixeles_nodata
   AGC_170     80780.081329      79086.206403          0.020969                12
   AGC_023     76682.271506      54724.101097          0.286353                 9
   AGC_164    117632.076023      52616.909325          0.552699                 8
   AGC_020     82856.226598      28792.252957          0.652503                 5
   AGC_044      6760.402971        133.478394          0.980256                 1


In [60]:
print("\n=== CENTROIDES SIN VALOR ===")

print(
    pendiente_comparacion[
        pendiente_comparacion["centroid"].isna()
    ][
        [
            "ID_POLIGON",
            "cobertura_valida",
            "n_pixeles",
            "n_pixeles_validos",
            "n_pixeles_nodata",
            "all_touched",
            "area_weighted"
        ]
    ].to_string(index=False)
)


=== CENTROIDES SIN VALOR ===
ID_POLIGON  cobertura_valida  n_pixeles  n_pixeles_validos  n_pixeles_nodata  all_touched  area_weighted
   AGC_023          0.286353         13                  4                 9     1.662428       1.766181
   AGC_170          0.020969         14                  2                12     1.363419       1.391080
   AGC_164          0.552699         16                  8                 8     0.827955       0.903925


In [61]:
import rasterio
import numpy as np

with rasterio.open(ELEV_PATH) as src_elev, rasterio.open(SLOPE_PATH) as src_slope:

    elev = src_elev.read(1)
    slope = src_slope.read(1)

    elev_nodata = src_elev.nodata
    slope_nodata = src_slope.nodata

    elev_valid = elev != elev_nodata
    slope_valid = slope != slope_nodata

    print("=== COMPARACIÓN ELEVACIÓN VS PENDIENTE ===")

    print("Dimensiones iguales:", elev.shape == slope.shape)
    print("Transform iguales:", src_elev.transform == src_slope.transform)
    print("CRS iguales:", src_elev.crs == src_slope.crs)

    print("\nElevación válida:", elev_valid.sum())
    print("Pendiente válida:", slope_valid.sum())

    print("\nPendiente NoData donde Elevación válida:",
          (elev_valid & ~slope_valid).sum())

    print("Pendiente válida donde Elevación NoData:",
          (~elev_valid & slope_valid).sum())

=== COMPARACIÓN ELEVACIÓN VS PENDIENTE ===
Dimensiones iguales: True
Transform iguales: True
CRS iguales: True

Elevación válida: 4081208
Pendiente válida: 4047765

Pendiente NoData donde Elevación válida: 33443
Pendiente válida donde Elevación NoData: 0


In [62]:
with rasterio.open(ELEV_PATH) as src_elev, rasterio.open(SLOPE_PATH) as src_slope:

    elev = src_elev.read(1)
    slope = src_slope.read(1)

    elev_valid = elev != src_elev.nodata
    slope_valid = slope != src_slope.nodata

    # Elevación válida pero pendiente NoData
    slope_missing = elev_valid & ~slope_valid

    print("Píxeles afectados:", slope_missing.sum())

    # Revisar sus coordenadas
    rows, cols = np.where(slope_missing)

    print("\n=== POSICIÓN DE LOS PIXELES ===")
    print("Fila mínima:", rows.min())
    print("Fila máxima:", rows.max())
    print("Columna mínima:", cols.min())
    print("Columna máxima:", cols.max())

    # Revisar si tienen vecinos NoData en elevación
    vecinos_nodata = []

    for r, c in zip(rows, cols):

        r0 = max(0, r - 1)
        r1 = min(elev.shape[0], r + 2)

        c0 = max(0, c - 1)
        c1 = min(elev.shape[1], c + 2)

        ventana = elev[r0:r1, c0:c1]

        nodata_count = np.sum(ventana == src_elev.nodata)

        vecinos_nodata.append(nodata_count)

    vecinos_nodata = np.array(vecinos_nodata)

    print("\n=== NODATA EN VENTANA 3x3 DE ELEVACIÓN ===")
    print("Sin vecinos NoData:", np.sum(vecinos_nodata == 0))
    print("Con al menos 1 NoData:", np.sum(vecinos_nodata > 0))
    print("Máximo NoData en ventana:", vecinos_nodata.max())

Píxeles afectados: 33443

=== POSICIÓN DE LOS PIXELES ===
Fila mínima: 54
Fila máxima: 3304
Columna mínima: 58
Columna máxima: 2837

=== NODATA EN VENTANA 3x3 DE ELEVACIÓN ===
Sin vecinos NoData: 0
Con al menos 1 NoData: 33443
Máximo NoData en ventana: 8


In [63]:
cobertura = pendiente_comparacion["cobertura_valida"]

print("=== COBERTURA DE PENDIENTE ===")

print("Media:", cobertura.mean())
print("Mediana:", cobertura.median())
print("Mínima:", cobertura.min())
print("Máxima:", cobertura.max())

print("\nParcelas con cobertura:")

print(">= 99%:", (cobertura >= 0.99).sum())
print(">= 95%:", (cobertura >= 0.95).sum())
print(">= 90%:", (cobertura >= 0.90).sum())
print(">= 75%:", (cobertura >= 0.75).sum())
print(">= 50%:", (cobertura >= 0.50).sum())
print("< 50%:", (cobertura < 0.50).sum())

print("\n=== PERCENTILES ===")
print(
    cobertura.quantile(
        [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    )
)

=== COBERTURA DE PENDIENTE ===
Media: 0.9872729959823988
Mediana: 0.9999999999999796
Mínima: 0.020968967816181197
Máxima: 1.0000000000016684

Parcelas con cobertura:
>= 99%: 192
>= 95%: 193
>= 90%: 193
>= 75%: 193
>= 50%: 195
< 50%: 2

=== PERCENTILES ===
0.00    0.020969
0.01    0.542045
0.05    1.000000
0.10    1.000000
0.25    1.000000
0.50    1.000000
0.75    1.000000
0.90    1.000000
0.95    1.000000
0.99    1.000000
1.00    1.000000
Name: cobertura_valida, dtype: float64


In [64]:
ids_problema = ["AGC_170", "AGC_023", "AGC_164", "AGC_020"]

print(
    pendiente_comparacion[
        pendiente_comparacion["ID_POLIGON"].isin(ids_problema)
    ][
        [
            "ID_POLIGON",
            "centroid",
            "all_touched",
            "area_weighted",
            "cobertura_valida",
            "n_pixeles_validos",
            "n_pixeles_nodata"
        ]
    ].sort_values("cobertura_valida").to_string(index=False)
)

ID_POLIGON  centroid  all_touched  area_weighted  cobertura_valida  n_pixeles_validos  n_pixeles_nodata
   AGC_170       NaN     1.363419       1.391080          0.020969                  2                12
   AGC_023       NaN     1.662428       1.766181          0.286353                  4                 9
   AGC_164       NaN     0.827955       0.903925          0.552699                  8                 8
   AGC_020  0.596809     0.511222       0.561396          0.652503                  8                 5


In [65]:
print("=== DISTRIBUCIÓN DE PENDIENTE ===")

print(
    pendiente_comparacion["area_weighted"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

=== DISTRIBUCIÓN DE PENDIENTE ===
count    197.000000
mean       2.792785
std        2.050182
min        0.281865
1%         0.317378
5%         0.440689
10%        0.622316
25%        1.149426
50%        2.126439
75%        3.891151
90%        5.856947
95%        7.274744
99%        7.997751
max        9.396395
Name: area_weighted, dtype: float64


In [66]:
comparacion_slope = pendiente_comparacion.copy()

comparacion_slope["diff_centroid"] = (
    comparacion_slope["centroid"]
    - comparacion_slope["area_weighted"]
).abs()

comparacion_slope["diff_all_touched"] = (
    comparacion_slope["all_touched"]
    - comparacion_slope["area_weighted"]
).abs()

print("=== CENTROID VS AREA-WEIGHTED ===")
print(
    comparacion_slope["diff_centroid"]
    .describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\n=== ALL-TOUCHED VS AREA-WEIGHTED ===")
print(
    comparacion_slope["diff_all_touched"]
    .describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

=== CENTROID VS AREA-WEIGHTED ===
count    194.000000
mean       0.396570
std        0.520286
min        0.000000
50%        0.219138
75%        0.515326
90%        0.858131
95%        1.350799
99%        2.644844
max        3.051843
Name: diff_centroid, dtype: float64

=== ALL-TOUCHED VS AREA-WEIGHTED ===
count    197.000000
mean       0.206343
std        0.272947
min        0.000000
50%        0.103753
75%        0.250483
90%        0.570316
95%        0.716386
99%        1.491669
max        1.599213
Name: diff_all_touched, dtype: float64


In [67]:
print("=== MAYORES DIFERENCIAS: CENTROID ===")

print(
    comparacion_slope[
        [
            "ID_POLIGON",
            "centroid",
            "area_weighted",
            "diff_centroid",
            "cobertura_valida"
        ]
    ]
    .sort_values("diff_centroid", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\n=== MAYORES DIFERENCIAS: ALL-TOUCHED ===")

print(
    comparacion_slope[
        [
            "ID_POLIGON",
            "all_touched",
            "area_weighted",
            "diff_all_touched",
            "cobertura_valida"
        ]
    ]
    .sort_values("diff_all_touched", ascending=False)
    .head(10)
    .to_string(index=False)
)

=== MAYORES DIFERENCIAS: CENTROID ===
ID_POLIGON  centroid  area_weighted  diff_centroid  cobertura_valida
   AGC_120  2.150894       5.202737       3.051843               1.0
   AGC_149  2.868600       5.869620       3.001020               1.0
   AGC_155  2.700266       5.318302       2.618035               1.0
   AGC_052  3.709572       6.077416       2.367844               1.0
   AGC_048  6.331957       4.350358       1.981599               1.0
   AGC_181  9.210482       7.264658       1.945823               1.0
   AGC_060  7.471274       9.396395       1.925121               1.0
   AGC_109  4.076930       2.382770       1.694160               1.0
   AGC_161  4.095114       5.733788       1.638675               1.0
   AGC_021  0.721116       2.079852       1.358737               1.0

=== MAYORES DIFERENCIAS: ALL-TOUCHED ===
ID_POLIGON  all_touched  area_weighted  diff_all_touched  cobertura_valida
   AGC_149     7.468833       5.869620          1.599213               1.0
   AGC_103 

In [75]:
topografia_qc = (
    pendiente_comparacion[
        [
            "ID_POLIGON",
            "area_weighted",
            "cobertura_valida"
        ]
    ]
    .rename(
        columns={
            "area_weighted": "slope_deg",
            "cobertura_valida": "slope_coverage"
        }
    )
    .merge(
        elevacion_comparacion[
            [
                "ID_POLIGON",
                "area_weighted",
                "cobertura_valida"
            ]
        ].rename(
            columns={
                "area_weighted": "elevation_m",
                "cobertura_valida": "elevation_coverage"
            }
        ),
        on="ID_POLIGON",
        how="inner"
    )
)

print("Shape:", topografia_qc.shape)

print("\n=== ELEVACIÓN ===")
print(
    topografia_qc["elevation_m"].describe(
        percentiles=[
            0.01, 0.05, 0.10,
            0.25, 0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)

print("\n=== PENDIENTE ===")
print(
    topografia_qc["slope_deg"].describe(
        percentiles=[
            0.01, 0.05, 0.10,
            0.25, 0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)

Shape: (197, 5)

=== ELEVACIÓN ===
count     197.000000
mean     2667.102482
std        92.283855
min      2496.648468
1%       2502.408115
5%       2532.002909
10%      2565.874770
25%      2611.351962
50%      2638.902931
75%      2726.683106
90%      2775.854218
95%      2880.829192
99%      2901.399907
max      2906.039520
Name: elevation_m, dtype: float64

=== PENDIENTE ===
count    197.000000
mean       2.792785
std        2.050182
min        0.281865
1%         0.317378
5%         0.440689
10%        0.622316
25%        1.149426
50%        2.126439
75%        3.891151
90%        5.856947
95%        7.274744
99%        7.997751
max        9.396395
Name: slope_deg, dtype: float64


In [73]:
from pathlib import Path

# Ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Carpeta de resultados intermedios de topografía
TOPO_DIR = PROJECT_ROOT / "data" / "interim" / "topography"
TOPO_DIR.mkdir(parents=True, exist_ok=True)

# Ruta de salida
TOPO_PATH = TOPO_DIR / "topografia_parcelas.csv"

# Guardar resultados ya auditados
topografia_qc.to_csv(
    TOPO_PATH,
    index=False
)

print(f"Archivo guardado en: {TOPO_PATH}")
print(f"Shape: {topografia_qc.shape}")
print("\nColumnas:")
print(topografia_qc.columns.tolist())

Archivo guardado en: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\interim\topography\topografia_parcelas.csv
Shape: (197, 5)

Columnas:
['ID_POLIGON', 'slope_deg', 'slope_coverage', 'elevation_m', 'elevation_coverage']


In [76]:
# Guardar resultados topográficos a nivel parcela
TOPO_DIR = DATA_INTERIM / "topography"
TOPO_DIR.mkdir(parents=True, exist_ok=True)

TOPO_PATH = TOPO_DIR / "topografia_parcelas.csv"

topografia_qc.to_csv(
    TOPO_PATH,
    index=False
)

print(f"Archivo guardado en: {TOPO_PATH}")
print(f"Shape: {topografia_qc.shape}")
print("\nColumnas:")
print(topografia_qc.columns.tolist())

NameError: name 'DATA_INTERIM' is not defined

In [ ]:
print("=== CONTROL DE VALORES ===")

print(
    "Elevación < 0 m:",
    (topografia_qc["elevation_m"] < 0).sum()
)

print(
    "Elevación NaN:",
    topografia_qc["elevation_m"].isna().sum()
)

print(
    "Pendiente < 0°:",
    (topografia_qc["slope_deg"] < 0).sum()
)

print(
    "Pendiente > 90°:",
    (topografia_qc["slope_deg"] > 90).sum()
)

print(
    "Pendiente NaN:",
    topografia_qc["slope_deg"].isna().sum()
)

=== CONTROL DE VALORES ===
Elevación < 0 m: 0
Elevación NaN: 0
Pendiente < 0°: 0
Pendiente > 90°: 0
Pendiente NaN: 0


In [ ]:
print("\n=== RELACIÓN ELEVACIÓN - PENDIENTE ===")

print(
    topografia_qc[
        ["elevation_m", "slope_deg"]
    ].corr()
)


=== RELACIÓN ELEVACIÓN - PENDIENTE ===
             elevation_m  slope_deg
elevation_m     1.000000   0.449593
slope_deg       0.449593   1.000000


In [ ]:
print("\n=== MAYOR ELEVACIÓN ===")

print(
    topografia_qc[
        ["ID_POLIGON", "elevation_m", "slope_deg"]
    ]
    .sort_values("elevation_m", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\n=== MAYOR PENDIENTE ===")

print(
    topografia_qc[
        ["ID_POLIGON", "elevation_m", "slope_deg"]
    ]
    .sort_values("slope_deg", ascending=False)
    .head(10)
    .to_string(index=False)
)


=== MAYOR ELEVACIÓN ===
ID_POLIGON  elevation_m  slope_deg
   AGC_088  2906.039520   2.064172
   AGC_165  2903.990381   2.364623
   AGC_013  2901.291970   2.852642
   AGC_006  2894.133470   2.412873
   AGC_126  2891.714403   2.082396
   AGC_072  2891.556591   3.452603
   AGC_190  2889.549618   2.154964
   AGC_172  2885.336642   3.674250
   AGC_012  2884.796555   5.512205
   AGC_132  2881.449670   3.324302

=== MAYOR PENDIENTE ===
ID_POLIGON  elevation_m  slope_deg
   AGC_060  2720.073969   9.396395
   AGC_057  2729.586381   8.243091
   AGC_142  2726.523945   7.987529
   AGC_191  2714.024148   7.826061
   AGC_028  2653.710429   7.699089
   AGC_118  2778.880946   7.544257
   AGC_119  2641.574050   7.488604
   AGC_009  2640.888224   7.428509
   AGC_049  2859.481228   7.417681
   AGC_134  2762.703709   7.315089


In [ ]:
topografia_qc = (
    pendiente_comparacion[
        ["ID_POLIGON", "area_weighted", "cobertura_valida"]
    ]
    .rename(
        columns={
            "area_weighted": "slope_deg",
            "cobertura_valida": "slope_coverage"
        }
    )
    .merge(
        elevacion_comparacion[
            ["ID_POLIGON", "area_weighted", "cobertura_valida"]
        ].rename(
            columns={
                "area_weighted": "elevation_m",
                "cobertura_valida": "elevation_coverage"
            }
        ),
        on="ID_POLIGON",
        how="inner"
    )
)

NameError: name 'pendiente_comparacion' is not defined

In [77]:
from pathlib import Path

# Ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Carpeta de resultados intermedios de topografía
TOPO_DIR = PROJECT_ROOT / "data" / "interim" / "topography"
TOPO_DIR.mkdir(parents=True, exist_ok=True)

# Ruta de salida
TOPO_PATH = TOPO_DIR / "topografia_parcelas.csv"

# Guardar resultados ya auditados
topografia_qc.to_csv(
    TOPO_PATH,
    index=False
)

print(f"Archivo guardado en: {TOPO_PATH}")
print(f"Shape: {topografia_qc.shape}")
print("\nColumnas:")
print(topografia_qc.columns.tolist())

Archivo guardado en: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\interim\topography\topografia_parcelas.csv
Shape: (197, 5)

Columnas:
['ID_POLIGON', 'slope_deg', 'slope_coverage', 'elevation_m', 'elevation_coverage']
